In [1]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.utils.data import DataLoader
from torch.utils.data import random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device: ', device)

Using device:  cuda


In [4]:
#unzip Dataset.zip

#224x224 res and normalizing pixels
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(
    root="Dataset",
    transform=transform
)

print(dataset.classes)
print(len(dataset))
print(dataset[0][0].shape)
print(dataset[0][1])

#80/20 training, validation split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size]
)

batch_size = 32

#shuffle for train, not for val
#send to gpu
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)

#check parameters
print('Number of training examples:', len(train_dataset))
print('Number of validation examples:', len(val_dataset))
print('Batch size:', batch_size)

print('Number of training batches:', len(train_loader))
print('Number of validation batches:', len(val_loader))

images, labels = next(iter(train_loader))

print("Shape of training images:", images.shape)
print("Shape of training labels:", labels.shape)

print("Minimum pixel value:", images.min())
print("Maximum pixel value:", images.max())

['Igneous', 'Metamorphic', 'Sedimentary']
2076
torch.Size([3, 224, 224])
0
Number of training examples: 1660
Number of validation examples: 416
Batch size: 32
Number of training batches: 52
Number of validation batches: 13
Shape of training images: torch.Size([32, 3, 224, 224])
Shape of training labels: torch.Size([32])
Minimum pixel value: tensor(-2.1179)
Maximum pixel value: tensor(2.6400)


In [5]:
#Load pretrained MobileNetV2, freeze pretrained layers
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

#Replace final classifier 3 classes (rock types)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    3
)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 102MB/s] 


In [6]:
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 5  # keep short

for epoch in range(epochs):
    #TRAIN
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    #VALIDATE
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} "
          f"Val Acc: {val_acc:.4f}")

Epoch [1/5] Train Loss: 0.9881 Val Acc: 0.5865
Epoch [2/5] Train Loss: 0.8668 Val Acc: 0.6779
Epoch [3/5] Train Loss: 0.8074 Val Acc: 0.7091
Epoch [4/5] Train Loss: 0.7668 Val Acc: 0.7163
Epoch [5/5] Train Loss: 0.7317 Val Acc: 0.7380
